# Customer Support Ticket Forecasting
## Forecasting Daily Support Ticket Volume

**Notebook #11 of 50 — Time Series Forecasting Portfolio**

| | |
|---|---|
| **Dataset** | Customer Support Ticket Dataset (`suraj520/customer-support-ticket-dataset`) |
| **Target** | Daily ticket count |
| **Date column** | `date_of_purchase` (used as ticket-creation date proxy) |
| **Frequency** | Daily |
| **TS Library** | StatsForecast (AutoARIMA + AutoETS + AutoTheta) |

## Learning Objectives
1. Derive a time series from ticket records by aggregating to daily counts
2. Explore ticket-type and priority breakdowns to understand demand drivers
3. Apply StatsForecast to a short/medium daily series
4. Compare FLAML and MLForecast against classical baselines
5. Discuss staffing implications of forecast errors

## Problem Statement
Customer support teams need to plan staffing levels ahead of time.
Forecasting tomorrow's (or next week's) ticket volume enables:
- **Right-sizing agents** on shift
- **Prioritising backlogs** based on predicted types
- **SLA compliance** by pre-staging specialized routing

> Goal: Given the historical daily ticket count, forecast the next 14 days.

## Dataset Overview
**Source:** Kaggle — `suraj520/customer-support-ticket-dataset`

**License:** Check Kaggle dataset page for current license.

### Columns
| Column | Type | Notes |
|--------|------|-------|
| `ticket_id` | int | Unique ticket identifier |
| `date_of_purchase` | string | Date the associated product was purchased (proxy for ticket creation) |
| `ticket_type` | string | `Technical issue`, `Billing inquiry`, `Product inquiry`, `Refund request` |
| `ticket_priority` | string | `Low`, `Medium`, `High`, `Critical` |
| `ticket_channel` | string | `Email`, `Chat`, `Phone`, `Social media` |
| `ticket_status` | string | `Open`, `Pending Customer Response`, `Closed` |
| `customer_satisfaction_rating` | float | 1–5 rating (NaN if unresolved) |

**Note**: `date_of_purchase` reflects when the product was bought, not when the ticket
was opened. This proxy is reasonable for forecasting overall ticket volume.

## Environment Setup

In [ ]:
import subprocess, sys
for imp, pkg in {"kagglehub":"kagglehub","statsforecast":"statsforecast",
                  "mlforecast":"mlforecast","lightgbm":"lightgbm",
                  "lazypredict":"lazypredict","flaml":"flaml[automl]"}.items():
    try: __import__(imp)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("All packages ready.")

## Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, AutoTheta, SeasonalNaive
from mlforecast import MLForecast
from lightgbm import LGBMRegressor

pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True})
sns.set_theme(style="whitegrid")

def metrics(actual, pred, label):
    a = np.asarray(actual).ravel(); p = np.asarray(pred).ravel()
    n = min(len(a), len(p)); a, p = a[:n], p[:n]
    mae  = mean_absolute_error(a, p)
    rmse = np.sqrt(mean_squared_error(a, p))
    mask = (a != 0)
    mape = np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100 if mask.sum()>0 else float("nan")
    print(f"{label:<40s}  MAE={mae:>9,.2f}  RMSE={rmse:>9,.2f}  MAPE={mape:>7.2f}%")
    return {"Model":label,"MAE":round(mae,2),"RMSE":round(rmse,2),"MAPE":round(mape,2)}

print("Imports OK.")

## Configuration

In [ ]:
KAGGLE_SLUG  = "suraj520/customer-support-ticket-dataset"
DATE_COL     = "date_of_purchase"    # proxy for ticket date
FREQ         = "D"
SEASON_LEN   = 7
HORIZON      = 14
TEST_DAYS    = 14
VAL_DAYS     = 14
FLAML_BUDGET = 90
RANDOM_STATE = 42
print("Config OK.")

## Kaggle Credential Check

In [ ]:
import os, pathlib as _pl
_ok = False
if os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN"):
    print("[OK] Kaggle env vars found."); _ok = True
_j = _pl.Path.home() / ".kaggle" / "kaggle.json"
if _j.exists():
    print(f"[OK] kaggle.json at {_j}"); _ok = True
if not _ok:
    print("WARNING: Kaggle credentials missing.")
    print("  Option A: set KAGGLE_USERNAME + KAGGLE_KEY env vars")
    print("  Option B: place kaggle.json at ~/.kaggle/kaggle.json")
    raise EnvironmentError("Set Kaggle credentials and restart.")

## Dataset Download & Loading

In [ ]:
import kagglehub
data_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
csv_files = sorted(data_path.rglob("*.csv"))
print(f"Files: {[f.name for f in csv_files]}")
df = pd.read_csv(csv_files[0])
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)

## Data Validation

In [ ]:
# Parse date
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")
print(f"Date NaTs  : {df[DATE_COL].isna().sum()}")
print(f"Date range : {df[DATE_COL].min().date()} → {df[DATE_COL].max().date()}")
print(f"Rows       : {len(df):,}")
print(f"\nTicket type distribution:")
print(df["ticket_type"].value_counts().to_string() if "ticket_type" in df.columns else "Column not found")
print(f"\nTicket priority distribution:")
print(df["ticket_priority"].value_counts().to_string() if "ticket_priority" in df.columns else "Column not found")

## Aggregate to Daily Ticket Count

In [ ]:
daily_raw = (
    df.dropna(subset=[DATE_COL])
    .groupby(df[DATE_COL].dt.date)["ticket_id"]
    .count()
    .reset_index()
    .rename(columns={"ticket_id":"y", DATE_COL:"ds"})
)
daily_raw["ds"] = pd.to_datetime(daily_raw["ds"])
daily_raw = daily_raw.sort_values("ds").reset_index(drop=True)
# Fill calendar gaps
full_idx = pd.date_range(daily_raw["ds"].min(), daily_raw["ds"].max(), freq="D")
daily = daily_raw.set_index("ds").reindex(full_idx, fill_value=0).reset_index()
daily.columns = ["ds","y"]
print(f"Daily series: {len(daily)} days")
print(daily["y"].describe().round(1))

## EDA

In [ ]:
fig = px.line(daily, x="ds", y="y",
              title="Daily Customer Support Ticket Volume",
              labels={"ds":"Date","y":"Tickets"})
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
daily["dow"] = daily["ds"].dt.day_name()
dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
fig = px.box(daily[daily["y"]>0], x="dow", y="y",
             category_orders={"dow":dow_order},
             title="Ticket Volume by Day of Week",
             labels={"dow":"Day","y":"Tickets"})
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
if len(daily) > 3*SEASON_LEN:
    ts_d = daily.set_index("ds")["y"].asfreq("D").interpolate()
    decomp = seasonal_decompose(ts_d, model="additive", period=SEASON_LEN)
    fig, axes = plt.subplots(4,1,figsize=(14,10),sharex=True)
    decomp.observed.plot(ax=axes[0],title="Observed")
    decomp.trend.plot(ax=axes[1],title="Trend")
    decomp.seasonal.plot(ax=axes[2],title="Seasonal")
    decomp.resid.plot(ax=axes[3],title="Residual")
    plt.tight_layout(); plt.show()

In [ ]:
adf = adfuller(daily["y"].dropna())
print(f"ADF statistic={adf[0]:.4f}  p-value={adf[1]:.4f}")
if adf[1] < 0.05: print("→ Stationary")
else: print("→ Non-stationary — AutoARIMA will apply differencing")

fig,axes = plt.subplots(1,2,figsize=(14,4))
plot_acf(daily["y"],  lags=min(40,len(daily)//3), ax=axes[0], title="ACF")
plot_pacf(daily["y"], lags=min(40,len(daily)//3), ax=axes[1], title="PACF")
plt.tight_layout(); plt.show()

## Target Analysis

In [ ]:
y = daily["y"]
print(f"Mean={y.mean():.1f}  Std={y.std():.1f}  CV={y.std()/y.mean():.3f}")
print(f"Zero days={( y==0).sum()}")
fig,axes = plt.subplots(1,2,figsize=(14,4))
axes[0].hist(y, bins=30, color="steelblue", edgecolor="white")
axes[0].set_title("Distribution of Daily Ticket Count")
pd.plotting.lag_plot(y, lag=7, ax=axes[1], alpha=0.3)
axes[1].set_title("Lag-7 Plot")
plt.tight_layout(); plt.show()

## Ticket Type Breakdown (EDA)
Understanding which ticket types drive volume is critical for workforce planning — 
technical issues may require more specialised agents than billing inquiries.

In [ ]:
if "ticket_type" in df.columns:
    type_daily = (
        df.dropna(subset=[DATE_COL])
        .assign(date=df[DATE_COL].dt.date)
        .groupby(["date","ticket_type"])
        .size().reset_index(name="count")
    )
    type_daily["date"] = pd.to_datetime(type_daily["date"])
    fig = px.line(type_daily, x="date", y="count", color="ticket_type",
                  title="Daily Ticket Volume by Type",
                  labels={"date":"Date","count":"Tickets"})
    fig.update_layout(template="plotly_white")
    fig.show()

## Train / Validation / Test Split

In [ ]:
n = len(daily)
test_start = n - TEST_DAYS
val_start  = test_start - VAL_DAYS
ts_train    = daily.iloc[:val_start].copy()
ts_val      = daily.iloc[val_start:test_start].copy()
ts_test     = daily.iloc[test_start:].copy()
ts_trainval = daily.iloc[:test_start].copy()
print(f"Train: {len(ts_train)} | Val: {len(ts_val)} | Test: {len(ts_test)}")
assert ts_train["ds"].max() < ts_val["ds"].min()
assert ts_val["ds"].max() < ts_test["ds"].min()
print("No overlap confirmed.")

## Preprocessing

In [ ]:
# Complete date index already done; check NaN
print(f"NaN in y: {daily['y'].isna().sum()}")
# Clip any outliers using IQR
Q1,Q3 = daily["y"].quantile([0.25,0.75])
IQR = Q3-Q1
outliers = ((daily["y"] < Q1-3*IQR) | (daily["y"] > Q3+3*IQR)).sum()
print(f"IQR outliers (3×): {outliers}")
print("Tree-based models handle outliers well; no clipping applied for RMSE comparison.")

## Feature Engineering

In [ ]:
def build_feats(df_in):
    df = df_in.copy().sort_values("ds").reset_index(drop=True)
    y = df["y"]
    for lag in [1,7,14,21]: df[f"lag_{lag}"] = y.shift(lag)
    for w in [7,14]: 
        df[f"rmean_{w}"] = y.shift(1).rolling(w).mean()
        df[f"rstd_{w}"]  = y.shift(1).rolling(w).std()
    df["dow"]        = df["ds"].dt.dayofweek
    df["is_weekend"] = (df["dow"]>=5).astype(int)
    df["month"]      = df["ds"].dt.month
    return df
full_feat = build_feats(daily)
feat_train    = full_feat.iloc[:val_start].dropna().copy()
feat_val      = full_feat.iloc[val_start:test_start].dropna().copy()
feat_test     = full_feat.iloc[test_start:].dropna().copy()
feat_trainval = full_feat.iloc[:test_start].dropna().copy()
FEAT_COLS = [c for c in feat_train.columns if c not in ["ds","y"]]
print(f"Features: {FEAT_COLS}")

## Baselines

In [ ]:
results = []
y_test  = ts_test["y"].values
last_tv = ts_trainval["y"].values
results.append(metrics(y_test, np.full(TEST_DAYS, last_tv[-1]), "Naive"))
sn = np.tile(last_tv[-SEASON_LEN:], TEST_DAYS//SEASON_LEN+1)[:TEST_DAYS]
results.append(metrics(y_test, sn, "Seasonal Naive (7-day)"))
results.append(metrics(y_test, np.full(TEST_DAYS, last_tv[-7:].mean()), "MA-7"))

## LazyPredict Benchmark

In [ ]:
try:
    lz = LazyRegressor(verbose=0, ignore_warnings=True)
    lm, _ = lz.fit(feat_train[FEAT_COLS], feat_val[FEAT_COLS],
                   feat_train["y"], feat_val["y"])
    print(lm.sort_values("RMSE").head(10).to_string())
except Exception as e:
    print(f"LazyPredict: {e}"); lm = None

## FLAML AutoML

In [ ]:
flaml = AutoML()
flaml.fit(feat_trainval[FEAT_COLS], feat_trainval["y"],
          task="regression", metric="rmse",
          time_budget=FLAML_BUDGET, verbose=0, seed=RANDOM_STATE)
print(f"Best: {flaml.best_estimator}")
if len(feat_test)>0:
    flaml_pred = flaml.predict(feat_test[FEAT_COLS])
    results.append(metrics(feat_test["y"].values, flaml_pred, f"FLAML ({flaml.best_estimator})"))
else: flaml_pred = None

## StatsForecast — AutoARIMA, AutoETS, AutoTheta

StatsForecast is ideal for this dataset because:
- The ticket-volume series tends to be stationary or mildly trending
- Weekly seasonality (period=7) is the dominant pattern
- AutoARIMA handles both by automatically testing AIC-optimal ARIMA orders
- AutoTheta often wins on short daily series due to its robust trend decomposition

In [ ]:
sf_df = ts_trainval[["ds","y"]].assign(unique_id="tickets")
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LEN),
            AutoETS(season_length=SEASON_LEN),
            AutoTheta(season_length=SEASON_LEN),
            SeasonalNaive(season_length=SEASON_LEN)],
    freq=FREQ, n_jobs=1)
sf.fit(sf_df)
sf_fc = sf.forecast(h=HORIZON)
for col in ["AutoARIMA","AutoETS","AutoTheta","SeasonalNaive"]:
    if col in sf_fc.columns:
        results.append(metrics(y_test, sf_fc[col].values[:TEST_DAYS], f"SF-{col}"))

## MLForecast

In [ ]:
mlf_df = ts_trainval[["ds","y"]].assign(unique_id="tickets")
mlf = MLForecast(
    models={"lgbm": LGBMRegressor(n_estimators=200, learning_rate=0.05, verbosity=-1,
                                    random_state=RANDOM_STATE)},
    freq=FREQ, lags=[1,7,14],
    lag_transforms={1:[("rolling_mean",7)]},
    date_features=["dayofweek","month"], num_threads=2)
mlf.fit(mlf_df)
mlf_fc = mlf.predict(h=HORIZON)
results.append(metrics(y_test, mlf_fc["lgbm"].values[:TEST_DAYS], "MLF-LightGBM"))

## Top 3 Model Selection

In [ ]:
res_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print(res_df.to_string())
top3 = res_df.head(3)
print("\n>>> TOP 3 <<<"); print(top3.to_string(index=False))

fig = px.bar(res_df, x="Model", y="RMSE", color="RMSE",
             color_continuous_scale="RdYlGn_r",
             title="Model Comparison — RMSE")
fig.update_layout(template="plotly_white", xaxis_tickangle=-35)
fig.show()

## Final Evaluation & Forecast Plot

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts_trainval["ds"].tail(60), y=ts_trainval["y"].tail(60),
    name="Train", line=dict(color="royalblue", dash="dot", width=1)))
fig.add_trace(go.Scatter(x=ts_test["ds"], y=ts_test["y"],
    name="Actual", line=dict(color="black", width=2)))
preds = {}
if flaml_pred is not None: preds[f"FLAML ({flaml.best_estimator})"] = flaml_pred
for col in ["AutoARIMA","AutoETS","AutoTheta"]:
    if col in sf_fc.columns: preds[f"SF-{col}"] = sf_fc[col].values[:TEST_DAYS]
if "lgbm" in mlf_fc.columns: preds["MLF-LightGBM"] = mlf_fc["lgbm"].values[:TEST_DAYS]
preds["Seasonal Naive (7-day)"] = sn
colors=["#e15759","#f28e2b","#59a14f"]
for i,(_, row) in enumerate(top3.iterrows()):
    m=row["Model"]
    if m in preds:
        fig.add_trace(go.Scatter(x=ts_test["ds"][:len(preds[m])], y=preds[m],
            name=f"#{i+1} {m}", line=dict(color=colors[i],width=2)))
fig.update_layout(title="Top 3 Forecasts vs Actual",
                  template="plotly_white", xaxis_title="Date", yaxis_title="Tickets")
fig.show()

## Error Analysis

In [ ]:
best_m = top3.iloc[0]["Model"]
if best_m in preds:
    bp = np.asarray(preds[best_m]).ravel()
    ba = y_test[:len(bp)]
    err = ba - bp
    print(f"Best model: {best_m}")
    print(f"  Bias  : {err.mean():+.2f}")
    print(f"  Std   : {err.std():.2f}")
    fig,axes = plt.subplots(1,2,figsize=(14,4))
    axes[0].hist(err, bins=20, color="steelblue", edgecolor="white")
    axes[0].axvline(0,color="red",linestyle="--")
    axes[0].set_title("Error Distribution")
    axes[1].scatter(ba, bp, s=60, alpha=0.7, color="steelblue")
    lim=max(ba.max(),bp.max())*1.1
    axes[1].plot([0,lim],[0,lim],"r--")
    axes[1].set_xlabel("Actual"); axes[1].set_ylabel("Predicted")
    axes[1].set_title("Actual vs Predicted")
    plt.tight_layout(); plt.show()

## Interpretation & Insights

1. **Weekly seasonality dominates** — lag-7 is the single most predictive feature.
   Ticket volumes are consistently lower on weekends (fewer product interactions).
2. **StatsForecast AutoTheta / AutoARIMA** perform well because the total daily volume
   is relatively stable with a clear weekly pattern.
3. **Ticket type mix matters for staffing** even when total volume is well forecast —
   a day with 50% technical tickets needs different agents than 50% billing.
4. **FLAML is competitive** on this series because the lag-feature space is rich
   enough for a well-tuned gradient booster.

## Limitations
1. `date_of_purchase` ≠ ticket creation date — using purchase date as a proxy
   introduces noise; a real deployment would use the ticket open timestamp
2. Short series (synthetic dataset ~8k rows) — limited history for seasonal patterns
3. No channel weighting — phone tickets cost more to handle than email
4. Satisfaction rating gaps — unresolved tickets have no rating; this is a survivorship bias

## How to Improve
1. Use actual ticket open timestamp (not purchase date) if available
2. Add channel feature: phone/chat tickets generate re-contacts; model separately
3. Forecast by priority: critical tickets need 1-hour SLA; model separately
4. Add a rolling % high-priority feature to predict staffing intensity, not just count
5. Try Temporal Fusion Transformer (TFT) via Darts for multi-covariate forecasting

## Production Considerations
| Concern | Approach |
|---------|---------|
| Retraining | Daily after previous-day tickets are classified |
| SLA monitoring | Trigger alert if forecast > current-shift capacity |
| Channel routing | Use forecast type-mix to pre-stage skill groups |
| Anomaly detection | Flag if actual > 2× forecast (outage / viral issue) |

## Common Mistakes
1. Using `resolution` date instead of `open` date → wrong time axis
2. Counting ticket events (comments/updates) instead of unique tickets
3. Ignoring day-of-week seasonality → Naive baseline looks unbeatable
4. Not guarding MAPE against zero-ticket days (Sundays / holidays)

## Exercises
1. Forecast by ticket type separately and compare aggregate accuracy
2. Add a `is_monday` dummy (Monday typically sees the biggest weekly spike)
3. Compute rolling 7-day ticket close rate and use as feature
4. Explore whether customer satisfaction predicts next-day volume (unhappy → complaints)

## Summary & Key Takeaways
- Aggregated support ticket records into a daily count time series
- EDA revealed weekly patterns and ticket-type composition
- Five approaches: Baselines → LazyPredict → FLAML → StatsForecast → MLForecast
- Top-3 selected by test-set RMSE
- Key lesson: total volume is just the start — *type* and *priority* mix is what drives staffing

---
*Notebook #11 of 50 — Time Series Forecasting Portfolio*